# LegalQA — Single-T4 Google Colab Smoke Runner

This notebook verifies QLoRA 4-bit loading, synchronous tensor materialization, single-GPU training arguments, PEFT adapter reload, and non-empty generation on a single Tesla T4 GPU before manual execution on Kaggle Dual-T4.

In [ ]:
# Step 1: Clone repository & install exact Kaggle user-space lock (preserving Colab Torch/CUDA)
!git clone https://github.com/silent9669/LegalQA.git
%cd LegalQA
!pip install --upgrade-strategy only-if-needed -r requirements-colab-smoke.txt

In [ ]:
# Step 2: Validate Single Tesla T4 & Deactivate Async Tensor Loading
import os
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
assert torch.cuda.device_count() == 1, "Colab smoke runner expects exactly 1 GPU (Tesla T4)."
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU Name: {gpu_name}")

In [ ]:
# Step 3: Setup minimal data directory
import os
os.makedirs("/content/legalqa-data", exist_ok=True)

# If using Google Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/LegalQA/*.parquet /content/legalqa-data/

# Or ensure qa_unique.parquet, retrieval_labels.parquet, legal_chunks.parquet are in /content/legalqa-data

In [ ]:
# Step 4: Run Generator Quick Smoke (3 steps)
!python scripts/run_colab_smoke.py \
    --data-root /content/legalqa-data \
    --component generator \
    --mode quick

In [ ]:
# Step 5: Run Generator Full Smoke (30 steps)
!python scripts/run_colab_smoke.py \
    --data-root /content/legalqa-data \
    --component generator \
    --mode full

In [ ]:
# Step 6: View Colab Smoke Test Report
import json
with open("/content/legalqa_colab_smoke/colab_smoke_report.json", "r", encoding="utf-8") as f:
    report = json.load(f)
print(json.dumps(report, indent=2))